# L2 step 4d, on the PC: the 3B's counts with no options against C4 and the blind rule

From saved files: the Kaggle run `l2-step4d-counts-without-options` (its `per_question.csv`, in `L2-step4-results/full/L2S4D/`), E7 step 3b's per-question counts (C4, the blind rule) and L2 step 4b's (the 3B with options). The rule and the expectation were fixed on 25 Nov in the Kaggle notebook's first cell, before the run:

1. the 3B's number, snapped to the nearest option, is right more often than the blind rule, beyond chance (exact McNemar, p < 0.05);
2. its number is closer to the true count than C4's raw count more often than not (sign test over the questions where the two differ, p < 0.05).

The 3B counts if both hold. Scored on the fair building-counting questions (largest option 20 or less) that both files hold, 2,000, and again on E5's clean ones.

In [1]:
# ── Cell 1: The three files, joined by question ──
import ast, json, math
from pathlib import Path
import numpy as np, pandas as pd

CT = Path.cwd()
RUN = CT / "L2-step4-results" / "full" / "L2S4D"
OUT = CT / "L2-step4-results" / "full" / "no-options"
OUT.mkdir(parents=True, exist_ok=True)
run_summary = json.loads((RUN / "summary.json").read_text(encoding="utf-8"))
assert not run_summary["smoke_test"], "this is the smoke test's output; use the full run's"
n = pd.read_csv(RUN / "per_question.csv").set_index("id")
e7 = pd.read_csv(CT / "E7-results" / "step3b" / "counting_per_question.csv").set_index("id")
ck = pd.read_csv(CT / "L2-step4-results" / "full" / "checks" / "counting_per_question.csv").set_index("id")
d = n.join(e7[["truth", "C4 blobs | count", "no_image_rule | correct", "e5_clean", "largest_option"]], how="inner") \
     .join(ck[ck["task"] == "BDC"][["correct_3b", "answer_c"]], how="inner")
assert len(d) == 2000, len(d)
assert (d["truth"] == d["truth_value"]).all(), "the files disagree on the true counts"
assert (d["largest_option"] <= 20).all()
d["values"] = d["values"].map(ast.literal_eval)
d["blind_right"] = d["no_image_rule | correct"].astype(bool)
d["with_options_right"] = d["correct_3b"].astype(bool)
d["nearest_right"] = d["nearest_right"].astype(bool)
d["c4_error"] = (d["C4 blobs | count"] - d["truth"]).abs()
print(f"{len(d):,} fair questions in all three files; the run: {run_summary['asked']:,} asked, "
      f"{run_summary['with_a_number_pct']}% with a number, {run_summary['minutes']} min")


2,000 fair questions in all three files; the run: 2,002 asked, 99.9% with a number, 59.9 min


In [2]:
# ── Cell 2: The two tests of the rule ──
def mcnemar(a, b):
    x, y = int((a & ~b).sum()), int((~a & b).sum())
    m = x + y
    return x, y, (1.0 if m == 0 else min(1.0, 2 * sum(math.comb(m, k) for k in range(min(x, y) + 1)) / 2 ** m))

def sign_test(wins, losses):
    m = wins + losses
    return 1.0 if m == 0 else min(1.0, 2 * sum(math.comb(m, k) for k in range(min(wins, losses) + 1)) / 2 ** m)

def tests(s):
    x, y, p1 = mcnemar(s["nearest_right"], s["blind_right"])
    e3 = s["abs_error"]                       # no number = no count: counted as a loss below
    closer = int((e3 < s["c4_error"]).sum())
    farther = int(((e3 > s["c4_error"]) | e3.isna()).sum())
    p2 = sign_test(closer, farther)
    return {"questions": len(s),
            "nearest_option_right_%": round(100 * s["nearest_right"].mean(), 2), "blind_rule_%": round(100 * s["blind_right"].mean(), 2),
            "only_3B_right": x, "only_rule_right": y, "test1_p": float(f"{p1:.3g}"),
            "test1_passes": bool(s["nearest_right"].mean() > s["blind_right"].mean() and p1 < 0.05),
            "3B_closer": closer, "C4_closer": farther, "ties": len(s) - closer - farther, "test2_p": float(f"{p2:.3g}"),
            "test2_passes": bool(closer > farther and p2 < 0.05)}

T = pd.DataFrame({"fair": tests(d), "fair, clean": tests(d[d["e5_clean"].astype(bool)])}).T
COUNTS = bool(T.loc["fair", "test1_passes"] and T.loc["fair", "test2_passes"])
print(T.to_string(), f"\n\nThe rule: the 3B {'counts' if COUNTS else 'does not count by this rule'} (both tests on the fair questions)")


            questions nearest_option_right_% blind_rule_% only_3B_right only_rule_right  test1_p test1_passes 3B_closer C4_closer ties test2_p test2_passes
fair             2000                   35.6         40.9           542             648  0.00232        False       824       801  375   0.585        False
fair, clean      1945                  35.22        40.87           523             633  0.00134        False       807       776  362   0.451        False 

The rule: the 3B does not count by this rule (both tests on the fair questions)


In [3]:
# ── Cell 3: How close each count is, and how often each is exactly right ──
def blind(vals):
    s = sorted(vals)
    return s[2] if s[4] > 20 else s[1]

def letters(text):
    import re
    t = str(text).upper().replace("THE ANSWER IS", "").replace("OPTION", "")
    return set(re.findall(r"[A-Z]", t))

def chosen(answer, vals):
    g = letters(answer)
    return vals["ABCDE".index(next(iter(g)))] if len(g) == 1 and next(iter(g)) in "ABCDE"[:len(vals)] else np.nan

counts = {"3B, no options (its number)": d["number"],
          "3B, with options (its pick)": pd.Series([chosen(a, v) for a, v in zip(d["answer_c"], d["values"])], index=d.index),
          "C4 raw count": d["C4 blobs | count"],
          "C4 nearest option": pd.Series([min(v, key=lambda x: abs(x - c)) for v, c in zip(d["values"], d["C4 blobs | count"])], index=d.index),
          "blind rule": pd.Series([blind(v) for v in d["values"]], index=d.index)}
E = pd.DataFrame([{"count from": k, "answered_%": round(100 * c.notna().mean(), 1),
                   "exact_%": round(100 * ((c - d["truth"]).abs() == 0).mean(), 1),
                   "within_1_%": round(100 * ((c - d["truth"]).abs() <= 1).mean(), 1),
                   "median_error": float((c - d["truth"]).abs().median()), "mean_error": round(float((c - d["truth"]).abs().mean()), 2)}
                  for k, c in counts.items()])
SAME_PICK = round(100 * float((d["nearest_letter"].fillna("") == d["answer_c"].astype(str).str.strip().str.upper().str[:1]).mean()), 1)
print(E.to_string(index=False), f"\n\nno-options number snapped to the option it picked with options: {SAME_PICK}% of questions")


                 count from  answered_%  exact_%  within_1_%  median_error  mean_error
3B, no options (its number)       100.0     21.9        40.2           2.0        4.12
3B, with options (its pick)       100.0     57.4        68.4           0.0        1.95
               C4 raw count       100.0     13.2        42.8           2.0        3.78
          C4 nearest option       100.0     42.2        54.9           1.0        2.46
                 blind rule       100.0     40.9        56.8           1.0        2.20 

no-options number snapped to the option it picked with options: 35.0% of questions


In [4]:
# ── Cell 4: Expectation (fixed 25 Nov before the run) against the result, and the files ──
s = run_summary
row = E.set_index("count from").loc["3B, no options (its number)"]
exp = pd.DataFrame([
    ("at least 95% of the answers contain a number", f"{s['with_a_number_pct']}%", s["with_a_number_pct"] >= 95),
    ("the exact count on 30–50% of questions", f"{row['exact_%']}%", 30 <= row["exact_%"] <= 50),
    ("snapped to the nearest option, right on 45–57% (test 1 passes)", f"{T.loc['fair', 'nearest_option_right_%']}%, test 1 {'passes' if T.loc['fair', 'test1_passes'] else 'fails'}",
     45 <= T.loc["fair", "nearest_option_right_%"] <= 57 and T.loc["fair", "test1_passes"]),
    ("a mean error of 2–3 buildings, below C4's raw 3.78 (test 2 passes)", f"{row['mean_error']}, test 2 {'passes' if T.loc['fair', 'test2_passes'] else 'fails'}",
     2 <= row["mean_error"] <= 3 and T.loc["fair", "test2_passes"]),
], columns=["expectation (25 Nov)", "result", "met"])

def md(df, index=True):
    t = df.reset_index() if index else df
    head = "| " + " | ".join(map(str, t.columns)) + " |\n|" + "---|" * len(t.columns)
    return head + "".join("\n| " + " | ".join(map(str, r)) + " |" for r in t.itertuples(index=False))

T.to_csv(OUT / "tests.csv")
E.to_csv(OUT / "count_error.csv", index=False)
exp.to_csv(OUT / "expectation.csv", index=False)
d.drop(columns=["values"]).to_csv(OUT / "per_question.csv")
readme = f'''# L2 step 4d: the 3B's counts with no options shown

The 3B (c) answered the benchmark's fair building-counting questions (largest option 20 or less) **with no answer options**, asked for a number ([Kaggle notebook](../../../l2-step4d-counts-without-options.ipynb), {s["asked"]:,} questions, {s["minutes"]} min on a T4). Compared on the PC ([notebook](../../../l2-step4d-compare.ipynb)) with C4's counts and the blind rule on the {len(d):,} questions all three files hold. The rule and the expectation were fixed on 25 Nov before the run.

## The rule

{md(T)}

**{"The 3B counts:" if COUNTS else "By the rule, the 3B does not count:"}** test 1 {"passes" if T.loc["fair", "test1_passes"] else "fails"} (its number, snapped to the nearest option, is right on {T.loc["fair", "nearest_option_right_%"]}% against the blind rule's {T.loc["fair", "blind_rule_%"]}%, p = {T.loc["fair", "test1_p"]}); test 2 {"passes" if T.loc["fair", "test2_passes"] else "fails"} (its number is closer to the truth than C4's raw count {T.loc["fair", "3B_closer"]} times against {T.loc["fair", "C4_closer"]}, p = {T.loc["fair", "test2_p"]}).

## How close the counts are

{md(E, index=False)}

With options the 3B was right on {round(100 * d["with_options_right"].mean(), 2)}% of these questions. Its number with no options snaps to the option it picked with options on {SAME_PICK}% of them.

## Expectation against result

{md(exp, index=False)}

## Files

`tests.csv`, `count_error.csv`, `expectation.csv`, `per_question.csv`; the run's own files are in `../L2S4D/`.
'''
(OUT / "README.md").write_text(readme, encoding="utf-8")
print(readme)


# L2 step 4d: the 3B's counts with no options shown

The 3B (c) answered the benchmark's fair building-counting questions (largest option 20 or less) **with no answer options**, asked for a number ([Kaggle notebook](../../../l2-step4d-counts-without-options.ipynb), 2,002 questions, 59.9 min on a T4). Compared on the PC ([notebook](../../../l2-step4d-compare.ipynb)) with C4's counts and the blind rule on the 2,000 questions all three files hold. The rule and the expectation were fixed on 25 Nov before the run.

## The rule

| index | questions | nearest_option_right_% | blind_rule_% | only_3B_right | only_rule_right | test1_p | test1_passes | 3B_closer | C4_closer | ties | test2_p | test2_passes |
|---|---|---|---|---|---|---|---|---|---|---|---|---|
| fair | 2000 | 35.6 | 40.9 | 542 | 648 | 0.00232 | False | 824 | 801 | 375 | 0.585 | False |
| fair, clean | 1945 | 35.22 | 40.87 | 523 | 633 | 0.00134 | False | 807 | 776 | 362 | 0.451 | False |

**By the rule, the 3B does not count:** te